# DC-Ops: YOLOv8n-seg -> QNN HTP (Snapdragon NPU)

Self-contained, all fixes baked in. **Runtime -> A100 GPU, then Runtime -> Run all.**

Cell 1 installs everything (~15 min). Cell 2 exports in a subprocess with the
linker path set correctly (this is what makes QNN HTP InitBackend succeed).
Cell 3 downloads the .pte. No manual fixes needed.

In [ ]:
# === CELL 1: Setup (libc++ + ExecuTorch + QNN SDK) — ~15 min ===
import os, subprocess

# libc++ is missing on Colab by default and every QNN lib needs it
!apt-get install -y libc++1 libc++abi1
!pip install -q ultralytics py-cpuinfo huggingface_hub

if not os.path.isdir('/content/executorch'):
    !git clone --depth 1 https://github.com/pytorch/executorch.git /content/executorch
%cd /content/executorch
!git submodule sync
!git submodule update --init --recursive --depth 1
!bash install_requirements.sh
!pip install -e . --no-build-isolation

# Download the Qualcomm QNN SDK (also pulls its libc++)
!python backends/qualcomm/scripts/download_qnn_sdk.py 2>&1 | tail -15

htp = subprocess.run('find / -name "libQnnHtp.so" 2>/dev/null', shell=True,
                     capture_output=True, text=True).stdout.strip()
print('\nlibQnnHtp.so at:', htp if htp else 'MISSING — re-run the download line above')

In [ ]:
# === CELL 2: Export YOLOv8n-seg -> QNN HTP .pte ===
# Runs in a subprocess so LD_LIBRARY_PATH is set BEFORE Python starts.
# Key fix: in-process os.environ changes don't reach the dynamic linker
# for QNN's transitive deps, but a subprocess env does.
import subprocess, os

found = [f for f in subprocess.run('find / -name "libQnnHtp.so" 2>/dev/null', shell=True,
         capture_output=True, text=True).stdout.strip().split('\n') if f]
assert found, 'libQnnHtp.so missing — re-run Cell 1 SDK download'
libdir = os.path.dirname(found[0])
qnn = libdir.split('/lib/')[0]
print('libdir:', libdir)
print('QNN_SDK_ROOT:', qnn)

script = r'''
import torch
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from executorch.backends.qualcomm.export_utils import build_executorch_binary, QnnConfig
from executorch.backends.qualcomm.quantizer.quantizer import QuantDtype

mp = hf_hub_download(repo_id="abhijitbetigeri/dc-ops-dataset",
                     filename="models/dc_ops_yolov8n_seg_v3.pt", repo_type="dataset")
core = YOLO(mp).model.eval()

class W(torch.nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, x):
        o = s.m(x)
        return o[0][0] if isinstance(o, (list, tuple)) else o   # detection tensor [1,52,8400]

w = W(core).eval()
calib = [(torch.randn(1,3,640,640),) for _ in range(20)]
cfg = QnnConfig(soc_model="SM8750", backend="htp",
                build_folder="/content/qnn_build", compile_only=True)
build_executorch_binary(model=w, qnn_config=cfg,
                        file_name="/content/dc_ops_yolo_qnn",
                        dataset=calib, quant_dtype=QuantDtype.use_8a8w)
print("EXPORT_OK")
'''
open('/content/run_export.py', 'w').write(script)

env = os.environ.copy()
env['QNN_SDK_ROOT'] = qnn
env['ADSP_LIBRARY_PATH'] = libdir
env['LD_LIBRARY_PATH'] = ':'.join([libdir, '/usr/lib/x86_64-linux-gnu',
                                   '/usr/lib/llvm-14/lib', env.get('LD_LIBRARY_PATH', '')])
env['PYTHONPATH'] = '/content/executorch:' + env.get('PYTHONPATH', '')

r = subprocess.run(['python', '/content/run_export.py'], env=env, cwd='/content/executorch',
                   capture_output=True, text=True)
print(r.stdout[-2500:])
print('\n--- STDERR (tail) ---\n', r.stderr[-2500:])

In [ ]:
# === CELL 3: Download the QNN .pte ===
import os
p = '/content/dc_ops_yolo_qnn.pte'
assert os.path.exists(p), 'No .pte produced — check Cell 2 STDERR output above'
print(f'{p}: {os.path.getsize(p)/1e6:.1f} MB')
from google.colab import files
files.download(p)